# Conciliacion: revision antes de enviar a el operador
Empresa contratista SRL - Ordenes de trabajo FTTH/IPTV, Cochabamba

Este cuaderno hace tres cosas, en orden. **Ejecuta las celdas de arriba hacia abajo.**

1. **Planilla:** subes la planilla de conciliacion y aplica reglas para detectar errores de tipeo y datos que no cuadran (numero de solicitud, decimales, tensores vs postes, tipo de trabajo vs servicio, nombres de tecnico, materiales fuera de lo normal).
2. **PDFs del sistema:** subes un ZIP con los PDFs descargados del sistema del operador y compara, orden por orden, lo que dice tu planilla contra lo que tiene el sistema. Tambien avisa de las ordenes con varios bloques IPTV, donde el operador puede no sumar bien.
3. **Informe:** descarga un Excel y un texto con todo lo que el operador podria observar, ordenado por gravedad, para corregir la planilla o preparar el respaldo antes de enviarla.

El sistema del operador no se puede modificar: cuando la diferencia viene del sistema, el informe te deja listo el detalle para justificarla.

## Paso 1 - Subir la planilla de conciliacion

In [ ]:
from google.colab import files

print("Selecciona tu planilla de conciliacion (.xlsx), NO el notebook .ipynb")
subido = files.upload()

candidatos = [n for n in subido.keys() if n.lower().endswith(".xlsx")]
if not candidatos:
    raise ValueError(
        f"No se subio ningun archivo .xlsx. Archivos recibidos: {list(subido.keys())}. "
        "Vuelve a correr esta celda y elige la planilla (.xlsx)."
    )
PATH_EXCEL = candidatos[0]
print(f"Planilla cargada: {PATH_EXCEL}")


In [ ]:
!pip install openpyxl pandas scikit-learn pdfplumber -q

import os
import re
import shutil
import subprocess
import zipfile
import difflib
import openpyxl
import pandas as pd
import numpy as np


## Configuracion
Posiciones de columnas de la hoja "Maestro Ejecutados", reglas y pesos. Si el operador cambia el formato de la planilla, solo hay que ajustar esta celda.

In [ ]:
COL_SOLICITUD = 1
COL_TIPO_TRABAJO = 7
COL_CANTIDAD_IPTV = 10
COL_TIPO_SERVICIO = 24
COL_TECNICO1 = 26
COL_TECNICO2 = 27
COL_AREA_COBERTURA = 3  # URBANO / RURAL
COL_POSTES = 81         # "Apoyos Instalados en Postes" -> cantidad de postes usados

# Bloque "MATERIAL INSTALADO": columnas 52 a 70 (19 materiales)
MATERIALES_COL_START = 52
MATERIALES = [
    "ANILLAS_GUIA", "TENSORES", "SOPORTE_MEDIO_TRAMO", "CABLE_DUCTO",
    "CINTA_ACERO", "CABLE_ACERO_CRUCE", "ACCESORIOS_POSTE", "CABLE_DROP",
    "ROSETA_OPTICA", "TERMINAL_SC", "CABLE_UTP", "CONECTOR_RJ45",
    "CAPUCHON_RJ45", "PATCH_CORD_SCSC", "ADAPTADOR_NAP", "CABLE_HDMI",
    "CABLE_RCA", "PATCH_CORD_UTP", "ADAPTADOR_ACDC",
]

# TENSORES se valida con una regla exacta contra la cantidad de postes,
# no con z-score, por eso se excluye del chequeo estadistico general.
MATERIALES_ESTADISTICOS = [m for m in MATERIALES if m != "TENSORES"]

# Tipos de servicio validos segun el tipo de trabajo declarado
SERVICIOS_DE_INSTALACION = {"SOLO ONT", "ONT + 1 STB", "ONT + 2 STB", "ONT + 3 STB"}
SERVICIOS_DE_ADICION = {"ADICION DE SERVICIO"}

# Materiales medidos en metros: el sistema los guarda como entero,
# un decimal en la conciliacion es señal de error de conversion/tipeo.
MATERIALES_METROS_ENTEROS = ["CABLE_DUCTO", "CABLE_DROP", "CABLE_UTP", "CINTA_ACERO", "CABLE_ACERO_CRUCE"]

# Posible FTTB (fibra hasta el edificio): sin postes ni materiales de tendido aereo.
# En esas ordenes es normal que falten o se desvien estos materiales, asi que la
# alerta baja a riesgo BAJO y se sugiere "posible FTTB" en lugar de marcar error.
MATERIALES_TENDIDO_AEREO = ["ANILLAS_GUIA", "TENSORES", "SOPORTE_MEDIO_TRAMO",
                            "CINTA_ACERO", "CABLE_ACERO_CRUCE", "ACCESORIOS_POSTE"]
# CABLE_DROP no se incluye: el drop siempre existe, un drop en 0 es error real aunque sea FTTB.
MATERIALES_AFECTADOS_POR_FTTB = MATERIALES_TENDIDO_AEREO + ["CABLE_DUCTO"]

# Formato esperado del numero de solicitud: solo se valida el primer digito
# y la cantidad de digitos (ajustar si cambia en otros meses)
SOLICITUD_PRIMER_DIGITO = "3"
SOLICITUD_LARGO = 9

# Umbral de similitud para detectar posible error de tipeo en nombres de tecnico
# (distancia de edicion maxima entre un nombre raro y uno frecuente)
TECNICO_DISTANCIA_MAXIMA = 2
TECNICO_FRECUENCIA_MINIMA_CONFIABLE = 3  # veces que debe aparecer un nombre para considerarlo "correcto"

# Materiales clave que NO deberian faltar si el perfil del grupo los usa siempre
# (media alta y desviacion baja -> son "obligatorios" para ese tipo de servicio)
UMBRAL_OBLIGATORIO_MEDIA = 0.9   # si >=90% de las ordenes del grupo lo usan
UMBRAL_ANOMALIA_ZSCORE = 2.0     # desviaciones estandar para marcar anomalia

# --- Pesos del indice de riesgo ---
# Alto (3): inconsistencias que suelen implicar que la orden esta mal cerrada
#           o que puede no coincidir con lo que el sistema del operador tiene registrado.
# Medio (2): inconsistencias fisicas/logicas detectables pero de menor gravedad.
# Bajo (1): errores de formato/redondeo, se corrigen facil y no cuestionan
#           si el trabajo se hizo bien.
PESOS = {
    "tipo_trabajo_vs_servicio": 3,
    "solicitud_formato": 3,
    "tensores_vs_postes": 2,
    "tecnico_tipeo": 2,
}

# Peso especifico por material para las alertas "material_faltante",
# "material_inusual" y "decimal_metros": depende de QUIEN asume el costo.
# - Materiales que paga/factura la empresa contratista (contratista): un error aqui
#   afecta directamente su costo/cobro -> peso alto.
# - Materiales que entrega la empresa de telecomunicaciones (operador): el error sigue
#   siendo una inconsistencia a corregir, pero no representa el mismo riesgo de
#   costo para el contratista -> peso bajo.
# Los marcados como "asumido" no fueron confirmados explicitamente por el usuario,
# se agruparon con el resto de herrajes de instalacion por similitud.
# Criterio: la lista de precios unitarios de la propia planilla del operador (hoja "Resumen").
# - Material con precio mayor a 0: la planilla del operador lo valora y lo descuenta si hay
#   diferencia -> afecta el costo/cobro de la contratista -> peso alto.
# - Material con precio 0 en esa planilla: el error sigue siendo una inconsistencia a corregir,
#   pero no tiene valor en Bs -> peso bajo.
MATERIALES_OPERADOR = {
    "CABLE_HDMI", "CABLE_RCA", "PATCH_CORD_UTP", "ADAPTADOR_ACDC",   # precio 0 en la planilla del operador
}
MATERIALES_CONTRATISTA = set(MATERIALES) - MATERIALES_OPERADOR

# Dos ejes: (1) quien paga el material y (2) que tan fuerte es la evidencia.
#   - "faltante": evidencia fuerte (un material casi obligatorio no esta)
#   - "decimal" / "inusual": evidencia mas debil (puede ser un caso legitimo,
#     por ejemplo una casa mas lejos que el promedio)
PESO_MATERIAL = {
    "contratista": {"faltante": 3, "decimal": 2, "inusual": 2},
    "operador":       {"faltante": 2, "decimal": 1, "inusual": 1},
}


def peso_material(nombre_material: str, tipo_alerta: str) -> int:
    """Peso segun quien paga el material y el tipo de alerta
    (tipo_alerta: 'faltante', 'decimal' o 'inusual')."""
    if nombre_material in MATERIALES_OPERADOR:
        quien = "operador"
    else:
        quien = "contratista"  # incluye los materiales asumidos y cualquier no clasificado
    return PESO_MATERIAL[quien][tipo_alerta]


NIVELES_RIESGO = ["BAJO", "MEDIO", "ALTO"]

# Precios unitarios (Bs) de la tabla de evaluacion de materiales (solo materiales del contratista)
PRECIOS_UNITARIOS = {
    "ANILLAS_GUIA": 4.70, "TENSORES": 4.70, "SOPORTE_MEDIO_TRAMO": 12.00,
    "CABLE_DUCTO": 9.30, "CINTA_ACERO": 15.00, "CABLE_ACERO_CRUCE": 8.80,
    "ACCESORIOS_POSTE": 205.00, "CABLE_DROP": 0.60, "ROSETA_OPTICA": 8.10,
    "TERMINAL_SC": 6.72, "CABLE_UTP": 6.00, "CONECTOR_RJ45": 4.80, "CAPUCHON_RJ45": 2.50,
    "PATCH_CORD_SCSC": 45.00, "ADAPTADOR_NAP": 18.36,
    # Precio 0 en la planilla del operador (sin valor en Bs)
    "CABLE_HDMI": 0.0, "CABLE_RCA": 0.0, "PATCH_CORD_UTP": 0.0, "ADAPTADOR_ACDC": 0.0,
}


## Leer la planilla

In [ ]:
def extraer_tabla(path_excel: str) -> pd.DataFrame:
    """Lee la hoja 'Maestro Ejecutados' y arma un DataFrame limpio, una fila por orden."""
    wb = openpyxl.load_workbook(path_excel, data_only=True)
    ws = wb["Maestro Ejecutados"]
    filas = list(ws.iter_rows(min_row=3, values_only=True))

    registros = []
    for r in filas:
        if r[COL_SOLICITUD] is None:
            continue  # fila vacia / de formato
        registro = {
            "solicitud": r[COL_SOLICITUD],
            "tipo_trabajo": r[COL_TIPO_TRABAJO],
            "tipo_servicio": r[COL_TIPO_SERVICIO],
            "cantidad_iptv": r[COL_CANTIDAD_IPTV] or 0,
            "area_cobertura": r[COL_AREA_COBERTURA],
            "tecnico1": r[COL_TECNICO1],
            "tecnico2": r[COL_TECNICO2],
            "postes": r[COL_POSTES] if isinstance(r[COL_POSTES], (int, float)) else 0,
        }
        materiales = r[MATERIALES_COL_START: MATERIALES_COL_START + len(MATERIALES)]
        for nombre, valor in zip(MATERIALES, materiales):
            registro[nombre] = valor if isinstance(valor, (int, float)) else 0
        registros.append(registro)

    df = pd.DataFrame(registros)
    return df

df = extraer_tabla(PATH_EXCEL)
print(f"Ordenes cargadas: {len(df)}")
df.head()


## Reglas de validacion sobre la planilla

In [ ]:
def construir_perfil_esperado(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula media, desviacion estandar y % de uso de cada material, por tipo_servicio."""
    stats = df.groupby("tipo_servicio")[MATERIALES_ESTADISTICOS].agg(["mean", "std"])
    uso = df.groupby("tipo_servicio")[MATERIALES_ESTADISTICOS].apply(lambda g: (g > 0).mean())
    return stats, uso


def validar_solicitud(solicitud):
    """Regla exacta: el numero de solicitud debe empezar con el digito esperado
    y tener el largo esperado. Detecta el caso de tipear el primer digito mal
    (ej. empezar con 2 en vez de 3). Devuelve (categoria, mensaje) o None."""
    texto = str(solicitud)
    if len(texto) != SOLICITUD_LARGO or not texto.startswith(SOLICITUD_PRIMER_DIGITO):
        msg = f"SOLICITUD='{texto}' no tiene el formato esperado (debe empezar con {SOLICITUD_PRIMER_DIGITO} y tener {SOLICITUD_LARGO} digitos)"
        return ("solicitud_formato", msg)
    return None


def validar_decimales(fila) -> list:
    """Regla exacta: los materiales medidos en metros se guardan como entero en el
    sistema; un valor con decimales en la conciliacion senala un error de tipeo/redondeo.
    Devuelve una lista de (categoria, mensaje)."""
    alertas = []
    for mat in MATERIALES_METROS_ENTEROS:
        valor = fila[mat]
        if valor and valor != int(valor):
            msg = f"{mat}={valor:g} tiene decimales (el sistema lo guarda como entero)"
            alertas.append(("decimal_metros", msg))
    return alertas


def construir_roster_tecnicos(df: pd.DataFrame) -> dict:
    """Cuenta cuantas veces aparece cada nombre de tecnico (tecnico1 + tecnico2)."""
    nombres = pd.concat([df["tecnico1"], df["tecnico2"]]).dropna().astype(str).str.strip()
    return nombres.value_counts().to_dict()


def validar_nombre_tecnico(nombre, roster: dict):
    """Si un nombre aparece pocas veces y es muy parecido a un nombre frecuente,
    es candidato a error de tipeo (ej. 'RUBEN GARCIA' en vez de 'RAUL GARCIA')."""
    if not nombre:
        return None
    nombre = str(nombre).strip()
    frecuencia = roster.get(nombre, 0)
    if frecuencia >= TECNICO_FRECUENCIA_MINIMA_CONFIABLE:
        return None  # nombre ya confiable, se repite bastante

    nombres_confiables = [n for n, f in roster.items() if f >= TECNICO_FRECUENCIA_MINIMA_CONFIABLE and n != nombre]
    parecidos = difflib.get_close_matches(nombre, nombres_confiables, n=1, cutoff=0.75)
    if parecidos:
        msg = f"TECNICO='{nombre}' (aparece {frecuencia}x) es muy similar a '{parecidos[0]}' — posible error de tipeo"
        return ("tecnico_tipeo", msg)
    return None


def validar_tensores_vs_postes(postes: float, tensores: float):
    """Regla exacta: por cada poste se usan entre 1 y 2 tensores (relacion fisica real)."""
    minimo_esperado = postes
    maximo_esperado = postes * 2
    if not (minimo_esperado <= tensores <= maximo_esperado):
        msg = (
            f"TENSORES={tensores:g} no coincide con POSTES={postes:g} "
            f"(esperado entre {minimo_esperado:g} y {maximo_esperado:g})"
        )
        return ("tensores_vs_postes", msg)
    return None


def es_posible_fttb(fila) -> bool:
    """Heuristica: instalacion nueva sin postes y sin ningun material de tendido aereo.
    (La planilla no distingue FTTH de FTTB, por eso es una sugerencia y no una certeza.)"""
    if fila["tipo_trabajo"] != "INSTALACION" or fila["postes"] != 0:
        return False
    return all(fila[m] == 0 for m in MATERIALES_TENDIDO_AEREO)


def validar_tipo_trabajo_vs_servicio(tipo_trabajo: str, tipo_servicio: str):
    """Regla exacta: el tipo de trabajo declarado debe ser consistente con el
    tipo de servicio ejecutado seleccionado (evita el caso de tipear 'ADICION'
    pero dejar seleccionado un tipo de servicio de instalacion nueva, o viceversa)."""
    if tipo_trabajo == "ADICION DE SERVICIO" and tipo_servicio not in SERVICIOS_DE_ADICION:
        msg = f"TIPO DE TRABAJO='ADICION DE SERVICIO' pero TIPO DE SERVICIO EJECUTADO='{tipo_servicio}'"
        return ("tipo_trabajo_vs_servicio", msg)
    if tipo_trabajo == "INSTALACION" and tipo_servicio not in SERVICIOS_DE_INSTALACION:
        msg = f"TIPO DE TRABAJO='INSTALACION' pero TIPO DE SERVICIO EJECUTADO='{tipo_servicio}'"
        return ("tipo_trabajo_vs_servicio", msg)
    return None


## Anomalias en la planilla (indice de riesgo)
Cada alerta pesa segun quien paga el material: un error en un material del contratista pesa mas que uno en un material del operador. Las ordenes sin postes ni materiales de tendido aereo se tratan como **posible FTTB**: sus alertas de material bajan a riesgo BAJO y se anota la sugerencia en la columna `nota_fttb`.

In [ ]:
def detectar_anomalias(df: pd.DataFrame) -> pd.DataFrame:
    """Marca, por orden, que materiales o campos se desvian de lo esperado,
    y calcula un indice de riesgo ponderado (no todas las alertas pesan igual)."""
    stats, uso = construir_perfil_esperado(df)
    roster = construir_roster_tecnicos(df)

    resultados = []
    for _, fila in df.iterrows():
        grupo = fila["tipo_servicio"]
        alertas = []            # lista de (categoria, mensaje) -- peso fijo por categoria
        alertas_con_peso = []   # lista de (peso, mensaje) -- peso especifico por material
        notas_fttb = []         # alertas de material que bajan a riesgo BAJO por posible FTTB
        posible_fttb = es_posible_fttb(fila)

        alerta_solicitud = validar_solicitud(fila["solicitud"])
        if alerta_solicitud:
            alertas.append(alerta_solicitud)

        alerta_tipo = validar_tipo_trabajo_vs_servicio(fila["tipo_trabajo"], grupo)
        if alerta_tipo:
            alertas.append(alerta_tipo)

        alerta_tensores = validar_tensores_vs_postes(fila["postes"], fila["TENSORES"])
        if alerta_tensores:
            alertas.append(alerta_tensores)

        # --- Materiales con decimales: peso segun quien paga el material ---
        for mat in MATERIALES_METROS_ENTEROS:
            valor = fila[mat]
            if valor and valor != int(valor):
                msg = f"{mat}={valor:g} tiene decimales (el sistema lo guarda como entero)"
                alertas_con_peso.append((peso_material(mat, "decimal"), msg))

        for campo in ("tecnico1", "tecnico2"):
            alerta_nombre = validar_nombre_tecnico(fila[campo], roster)
            if alerta_nombre:
                alertas.append(alerta_nombre)

        for mat in MATERIALES_ESTADISTICOS:
            media = stats.loc[grupo, (mat, "mean")]
            std = stats.loc[grupo, (mat, "std")] or 0
            pct_uso = uso.loc[grupo, mat]
            valor = fila[mat]

            if pct_uso >= UMBRAL_OBLIGATORIO_MEDIA and valor == 0:
                msg = f"Falta {mat} (se usa en {pct_uso:.0%} de '{grupo}')"
                if posible_fttb and mat in MATERIALES_AFECTADOS_POR_FTTB:
                    notas_fttb.append(msg)
                else:
                    alertas_con_peso.append((peso_material(mat, "faltante"), msg))
            elif std > 0:
                z = abs(valor - media) / std
                if z >= UMBRAL_ANOMALIA_ZSCORE:
                    msg = f"{mat}={valor:g} inusual (esperado ~{media:.1f}, grupo '{grupo}')"
                    if posible_fttb and mat in MATERIALES_AFECTADOS_POR_FTTB:
                        notas_fttb.append(msg)
                    else:
                        alertas_con_peso.append((peso_material(mat, "inusual"), msg))

        puntaje = sum(PESOS[categoria] for categoria, _ in alertas) + sum(p for p, _ in alertas_con_peso)
        todas_las_alertas_texto = [m for _, m in alertas] + [m for _, m in alertas_con_peso]

        resultados.append({
            "solicitud": fila["solicitud"],
            "tipo_trabajo": fila["tipo_trabajo"],
            "tipo_servicio": grupo,
            "area_cobertura": fila["area_cobertura"],
            "cantidad_iptv": fila["cantidad_iptv"],
            "tecnico1": fila["tecnico1"],
            "posible_fttb": posible_fttb,
            "nota_fttb": ("POSIBLE FTTB (sin postes ni materiales de tendido aereo), riesgo reducido: "
                          + " | ".join(notas_fttb)) if notas_fttb else "",
            "n_alertas": len(todas_las_alertas_texto),
            "puntaje_riesgo": puntaje,
            "detalle_alertas": " | ".join(todas_las_alertas_texto) if todas_las_alertas_texto else "",
        })

    resultado_df = pd.DataFrame(resultados)
    resultado_df["nivel_riesgo"] = asignar_nivel_riesgo(resultado_df["puntaje_riesgo"])
    return resultado_df.sort_values("puntaje_riesgo", ascending=False)


def asignar_nivel_riesgo(puntajes: pd.Series) -> pd.Series:
    """Convierte el puntaje ponderado en BAJO / MEDIO / ALTO.

    Los cortes se derivan directamente del significado de los pesos (no son
    arbitrarios): una sola alerta de peso bajo/medio (1 o 2) deja la orden en
    MEDIO; alcanzar o superar 3 -- el peso de una alerta grave por si sola
    (material faltante, inconsistencia de tipo de trabajo, solicitud mal
    formada), o varias alertas menores combinadas -- la sube a ALTO.
    """
    def clasificar(p):
        if p == 0:
            return "BAJO"
        elif p <= 2:
            return "MEDIO"
        else:
            return "ALTO"

    return puntajes.apply(clasificar)


anomalias = detectar_anomalias(df)
print(f"Ordenes con al menos una alerta: {(anomalias['n_alertas'] > 0).sum()} de {len(df)}")
print(anomalias["nivel_riesgo"].value_counts().to_string())
anomalias[anomalias["n_alertas"] > 0].head(15)


## Resumen por tecnico (instalaciones y adiciones)

In [ ]:
def resumen_por_tecnico(df: pd.DataFrame) -> pd.DataFrame:
    """Cuadro simple: nombre de tecnico, cuantas instalaciones y cuantas adiciones hizo
    (cuenta al tecnico tanto si aparece en TECNICO1 como en TECNICO2 de la orden)."""
    filas = []
    for _, fila in df.iterrows():
        es_adicion = fila["tipo_trabajo"] == "ADICION DE SERVICIO"
        for campo in ("tecnico1", "tecnico2"):
            nombre = fila[campo]
            if not nombre:
                continue
            filas.append({"tecnico": str(nombre).strip(), "es_adicion": es_adicion})

    detalle = pd.DataFrame(filas)
    resumen = detalle.groupby("tecnico")["es_adicion"].agg(
        instalaciones=lambda s: (~s).sum(),
        adiciones=lambda s: s.sum(),
    ).reset_index()
    resumen["total"] = resumen["instalaciones"] + resumen["adiciones"]
    return resumen.sort_values("total", ascending=False)

resumen_tec = resumen_por_tecnico(df)
resumen_tec


## Paso 2 - Comparar con los PDFs del sistema
Descarga del sistema del operador los PDFs de las ordenes ejecutadas (los que dejan seleccionar el texto) y juntalos en un **ZIP**. No importa como se llamen los archivos: el numero de solicitud se lee de adentro del PDF.

El capuchon RJ45 no aparece en el sistema, asi que no se compara. Los PDFs escaneados (solo imagen) no se pueden leer y quedan listados aparte.

In [ ]:
# === PARSER ===
# Nombre del material en el PDF -> nombre de columna en la planilla
MAPA_MATERIALES = {
    "SOPORTE MEDIO TRAMO": "SOPORTE_MEDIO_TRAMO",
    "CAIDA TENSORIAL": "TENSORES",
    "TENSOR DROP": "TENSORES",
    "CINTA ACERO": "CINTA_ACERO",
    "ANILLAS GUIA": "ANILLAS_GUIA",
    "CAIDA DE CABLE": "CABLE_DROP",
    "CABLE DROP": "CABLE_DROP",
    "CABLE DUCTO": "CABLE_DUCTO",
    "TERMINAL SC": "TERMINAL_SC",
    "ROSETA OPTICA": "ROSETA_OPTICA",
    "CABLE UTP": "CABLE_UTP",
    "CONECTOR RJ45": "CONECTOR_RJ45",
    "CONECTOR RJ-45": "CONECTOR_RJ45",
    "PACH CORD SC-SC": "PATCH_CORD_SCSC",
    "CORDON PACH SC-SC": "PATCH_CORD_SCSC",
    "ADAPTADOR AC/DC": "ADAPTADOR_ACDC",
    "PACH CORD UTP": "PATCH_CORD_UTP",
    "CABLE HDMI": "CABLE_HDMI",
    "CABLE RCA": "CABLE_RCA",
}

# Materiales que el sistema del operador NO registra en el PDF (no se pueden comparar).
# El capuchon RJ45 no aparece en ninguno de los PDFs revisados.
NO_ESTAN_EN_SISTEMA = {"CAPUCHON_RJ45"}

_LINEA_MATERIAL = re.compile(
    r"^\s*([A-Z][A-Z0-9 /\-]+?)\s+(\d+(?:\.\d+)?)\s+(\d+(?:\.\d+)?)\s+(\d+(?:\.\d+)?)\s*$"
)


def texto_pdf(path: str) -> str:
    """Texto del PDF. Usa pdftotext si existe; si no, pdfplumber."""
    try:
        r = subprocess.run(["pdftotext", "-layout", path, "-"], capture_output=True, text=True)
        if r.returncode == 0:
            return r.stdout
    except FileNotFoundError:
        pass
    import pdfplumber
    with pdfplumber.open(path) as pdf:
        return "\n".join((p.extract_text() or "") for p in pdf.pages)


def _normalizar(nombre: str) -> str:
    return re.sub(r"\s+", " ", nombre).strip()


def leer_orden_pdf(path: str):
    """Devuelve {'solicitud', 'materiales', 'bloques_iptv'} o None si el PDF no tiene
    texto (escaneado). Las ordenes con IPTV traen un bloque de materiales por cada STB
    (seccion M); 'materiales' es la suma de la seccion L y de todos los bloques."""
    t = texto_pdf(path)
    if len(t.strip()) < 200:
        return None
    sol = re.search(r"Request\s*Number:?\s*(\d{9})", t, re.S) or re.search(r"\b(3\d{8})\b", t)

    ini, fin = t.find("L. Installed materials"), t.find("N. Field Engineer")
    zona = t[ini:fin] if 0 <= ini < fin else t
    pos_m = zona.find("M. IPTV materials installed")
    parte_l = zona if pos_m < 0 else zona[:pos_m]
    parte_m = "" if pos_m < 0 else zona[pos_m:]

    materiales = {}
    for linea in parte_l.splitlines():
        m = _LINEA_MATERIAL.match(linea)
        if m and _normalizar(m.group(1)) in MAPA_MATERIALES:
            col = MAPA_MATERIALES[_normalizar(m.group(1))]
            materiales[col] = materiales.get(col, 0) + float(m.group(3))

    # Bloques IPTV: un bloque nuevo empieza cuando se repite un nombre de material
    bloques, actual, vistos = [], {}, set()
    for linea in parte_m.splitlines():
        m = _LINEA_MATERIAL.match(linea)
        if not m:
            continue
        nombre = _normalizar(m.group(1))
        if nombre in vistos:
            bloques.append(actual)
            actual, vistos = {}, set()
        vistos.add(nombre)
        if nombre in MAPA_MATERIALES:
            col = MAPA_MATERIALES[nombre]
            actual[col] = actual.get(col, 0) + float(m.group(3))
    if vistos:
        bloques.append(actual)
    for b in bloques:
        for col, v in b.items():
            materiales[col] = materiales.get(col, 0) + v

    return {"solicitud": int(sol.group(1)) if sol else None,
            "materiales": materiales, "bloques_iptv": bloques}


# === COMPARACION ===
def describir_diferencia(p: float, s: float):
    """Clasifica una diferencia planilla (p) vs sistema (s). Devuelve (tipo, detalle)."""
    if p != int(p) and abs(p - s) < 1:
        return "REDONDEO", "La planilla tiene decimales y el sistema guarda numeros enteros"
    if p == 0:
        return "FALTA EN PLANILLA", f"El sistema registra {s:g} y la planilla 0"
    if s == 0:
        return "SOBRA EN PLANILLA", f"La planilla tiene {p:g} y el sistema no lo registra"
    if abs(p - s) == 1:
        return "DIFERENCIA DE 1", "Posible error de tipeo o de conteo"
    pe, se = str(int(p)), str(int(s))
    if len(pe) == len(se) and sum(a != b for a, b in zip(pe, se)) == 1:
        return "UN DIGITO DISTINTO", "Posible error de tipeo"
    if abs(len(pe) - len(se)) == 1:
        largo, corto = (pe, se) if len(pe) > len(se) else (se, pe)
        if any(largo[:i] + largo[i + 1:] == corto for i in range(len(largo))):
            return "DIGITO DE MAS O DE MENOS", "Posible error de tipeo"
    return "CANTIDAD DISTINTA", "Las cantidades no coinciden"


def comparar_orden(fila_planilla, orden: dict, materiales: list, precios: dict, materiales_operador: set):
    """Lista de hallazgos (dicts) de una orden: planilla vs PDF del sistema."""
    difs = []
    for mat in materiales:
        if mat in NO_ESTAN_EN_SISTEMA:
            continue
        p, s = float(fila_planilla[mat]), float(orden["materiales"].get(mat, 0))
        if abs(p - s) > 1e-9:
            difs.append((mat, p, s))

    # Materiales intercambiados (ej. ducto y drop cargados al reves en el sistema)
    cambiados = {}
    for i in range(len(difs)):
        for j in range(i + 1, len(difs)):
            a, b = difs[i], difs[j]
            if a[1] == b[2] and b[1] == a[2]:
                cambiados[a[0]], cambiados[b[0]] = b[0], a[0]

    hallazgos = []
    for mat, p, s in difs:
        tipo, detalle = describir_diferencia(p, s)
        if mat in cambiados:
            tipo = "POSIBLE CAMBIO DE MATERIAL"
            detalle = f"Los valores de {mat} y {cambiados[mat]} parecen intercambiados en el sistema"
        precio = precios.get(mat, 0.0)
        bs = round((p - s) * precio, 2)
        if precio == 0:
            efecto = "Sin efecto en el monto (material del operador)"
        elif bs > 0:
            efecto = f"Facturas {bs:.2f} Bs de mas (riesgo de multa)"
        elif bs < 0:
            efecto = f"Dejas de facturar {-bs:.2f} Bs"
        else:
            efecto = "Sin efecto en el monto"
        gravedad = "BAJA" if tipo == "REDONDEO" else ("MEDIA" if mat in materiales_operador else "ALTA")
        hallazgos.append({
            "solicitud": orden["solicitud"], "material": mat, "planilla": p, "sistema": s,
            "tipo": tipo, "detalle": detalle, "bs": bs, "efecto": efecto, "gravedad": gravedad,
        })
    return hallazgos


def alertas_iptv(fila_planilla, orden: dict):
    """Avisos preventivos sobre los bloques IPTV del sistema (uno por STB)."""
    avisos = []
    bloques = orden.get("bloques_iptv", [])
    n, esperado = len(bloques), int(fila_planilla["cantidad_iptv"] or 0)
    if n != esperado:
        avisos.append(("ALTA", f"El sistema tiene {n} bloque(s) de materiales IPTV y la planilla declara "
                               f"{esperado} IPTV. Revisar el cierre de la orden."))
    if n >= 2:
        partes = []
        for col, nombre in (("CABLE_UTP", "UTP"), ("CONECTOR_RJ45", "RJ45"), ("CABLE_HDMI", "HDMI")):
            vals = [b.get(col, 0) for b in bloques]
            if any(vals):
                partes.append(f"{nombre}: " + " + ".join(f"{v:g}" for v in vals) + f" = {sum(vals):g}")
        avisos.append(("MEDIA", f"PREVENTIVA: el sistema tiene {n} bloques IPTV (uno por STB). Si el operador compara "
                                f"contra un solo bloque, vera cifras menores a la suma. Totales en el sistema -> "
                                + "; ".join(partes) + ". Tener a mano el PDF para defender la suma."))
    return avisos


# === PROCESAR ===
def preparar_pdfs(archivos_subidos, carpeta="pdfs_sistema"):
    """Junta en una carpeta los PDFs subidos, descomprimiendo los .zip."""
    if os.path.isdir(carpeta):
        shutil.rmtree(carpeta)
    os.makedirs(carpeta)
    for nombre in archivos_subidos:
        if nombre.lower().endswith(".zip"):
            with zipfile.ZipFile(nombre) as z:
                z.extractall(carpeta)
        elif nombre.lower().endswith(".pdf"):
            shutil.copy(nombre, carpeta)
    return carpeta


def procesar_pdfs(carpeta, df, materiales, precios, materiales_operador):
    """Lee todos los PDFs de la carpeta y los compara con la planilla.
    Devuelve (hallazgos_sistema, alertas_iptv_df, info_pdf)."""
    df = df.copy()
    df["solicitud"] = df["solicitud"].astype(int)
    por_solicitud = df.drop_duplicates("solicitud").set_index("solicitud", drop=False)

    rutas = []
    for raiz, _, nombres in os.walk(carpeta):
        rutas += [os.path.join(raiz, n) for n in nombres if n.lower().endswith(".pdf")]

    hallazgos, avisos = [], []
    leidos, no_leidos, fuera = set(), [], []
    for ruta in sorted(rutas):
        orden = leer_orden_pdf(ruta)
        if orden is None or orden["solicitud"] is None:
            no_leidos.append(os.path.basename(ruta))
            continue
        sol = orden["solicitud"]
        if sol not in por_solicitud.index:
            fuera.append(sol)
            continue
        leidos.add(sol)
        fila = por_solicitud.loc[sol]
        hallazgos += comparar_orden(fila, orden, materiales, precios, materiales_operador)
        for gravedad, texto in alertas_iptv(fila, orden):
            avisos.append({"solicitud": sol, "gravedad": gravedad, "descripcion": texto})

    info = {"pdfs_en_carpeta": len(rutas), "ordenes_leidas": len(leidos), "no_leidos": no_leidos,
            "fuera_de_planilla": fuera, "ordenes_planilla": len(por_solicitud),
            "ordenes_sin_pdf": len(por_solicitud) - len(leidos), "solicitudes_leidas": sorted(leidos)}
    return pd.DataFrame(hallazgos), pd.DataFrame(avisos), info


In [ ]:
from google.colab import files

print("Sube el ZIP con los PDFs del sistema (puedes subir varios zip o PDFs sueltos)")
subidos_pdf = files.upload()
# Si el ZIP pesa mucho: subelo desde el icono de carpeta (panel izquierdo) y usa:
# CARPETA_PDFS = preparar_pdfs(["nombre_del_archivo.zip"])
CARPETA_PDFS = preparar_pdfs(list(subidos_pdf.keys()))
print("PDFs listos en:", CARPETA_PDFS)


In [ ]:
hallazgos_sistema, alertas_iptv_df, info_pdf = procesar_pdfs(
    CARPETA_PDFS, df, MATERIALES, PRECIOS_UNITARIOS, MATERIALES_OPERADOR
)
print(f"PDFs en la carpeta: {info_pdf['pdfs_en_carpeta']} | ordenes leidas y comparadas: {info_pdf['ordenes_leidas']}")
print(f"Ordenes de la planilla sin PDF (no comparadas): {info_pdf['ordenes_sin_pdf']}")
if info_pdf["no_leidos"]:
    print("PDFs escaneados o ilegibles:", info_pdf["no_leidos"])
if info_pdf["fuera_de_planilla"]:
    print("PDFs cuya solicitud no esta en la planilla:", info_pdf["fuera_de_planilla"])
print(f"\nDiferencias planilla vs sistema: {len(hallazgos_sistema)} en {hallazgos_sistema['solicitud'].nunique() if len(hallazgos_sistema) else 0} ordenes")
hallazgos_sistema


## Paso 3 - Informe para descargar

In [ ]:
# === INFORME ===
_ORDEN_GRAVEDAD = {"ALTA": 0, "MEDIA": 1, "BAJA": 2}
_ACCION = {
    "REDONDEO": "Poner el valor entero en la planilla (el sistema guarda enteros).",
    "POSIBLE CAMBIO DE MATERIAL": "Verificar con la orden fisica. Si el error esta en el sistema, preparar la "
                                  "justificacion para el operador (el sistema no se puede modificar).",
}
_ACCION_DEFECTO = ("Verificar con la orden fisica. Si el error es de la planilla, corregirla; si es del "
                   "sistema, preparar la justificacion para el operador.")


def construir_hallazgos(anomalias, hallazgos_sistema, alertas_iptv_df):
    """Tabla unica con todo lo que se podria observar, una fila por hallazgo."""
    filas = []
    a = anomalias[anomalias["nivel_riesgo"] != "BAJO"]
    for _, r in a.iterrows():
        filas.append({"solicitud": r["solicitud"], "origen": "Reglas sobre la planilla",
                      "gravedad": "ALTA" if r["nivel_riesgo"] == "ALTO" else "MEDIA", "material": "",
                      "planilla": "", "sistema": "", "tipo": "", "bs": "",
                      "descripcion": r["detalle_alertas"],
                      "accion_sugerida": "Revisar contra la orden fisica antes de enviar."})
    if "nota_fttb" in anomalias.columns:
        for _, r in anomalias[anomalias["nota_fttb"] != ""].iterrows():
            filas.append({"solicitud": r["solicitud"], "origen": "Posible FTTB (riesgo reducido)",
                          "gravedad": "BAJA", "material": "", "planilla": "", "sistema": "", "tipo": "",
                          "bs": "", "descripcion": r["nota_fttb"],
                          "accion_sugerida": "Confirmar que es FTTB (sin tendido aereo). Si no lo es, revisar contra la orden fisica."})
    if len(hallazgos_sistema):
        for _, r in hallazgos_sistema.iterrows():
            filas.append({"solicitud": r["solicitud"], "origen": "Planilla vs sistema (PDF)",
                          "gravedad": r["gravedad"], "material": r["material"], "planilla": r["planilla"],
                          "sistema": r["sistema"], "tipo": r["tipo"], "bs": r["bs"],
                          "descripcion": f"{r['detalle']}. {r['efecto']}",
                          "accion_sugerida": _ACCION.get(r["tipo"], _ACCION_DEFECTO)})
    if len(alertas_iptv_df):
        for _, r in alertas_iptv_df.iterrows():
            filas.append({"solicitud": r["solicitud"], "origen": "Aviso preventivo IPTV",
                          "gravedad": r["gravedad"], "material": "", "planilla": "", "sistema": "",
                          "tipo": "", "bs": "", "descripcion": r["descripcion"],
                          "accion_sugerida": "Guardar el PDF del sistema como respaldo."})
    h = pd.DataFrame(filas, columns=["solicitud", "origen", "gravedad", "material", "planilla", "sistema",
                                     "tipo", "bs", "descripcion", "accion_sugerida"])
    if len(h):
        h["_o"] = h["gravedad"].map(_ORDEN_GRAVEDAD)
        h = h.sort_values(["_o", "solicitud"]).drop(columns="_o").reset_index(drop=True)
    return h


def resumen_general(df, anomalias, hallazgos_sistema, alertas_iptv_df, info_pdf):
    hs = hallazgos_sistema if len(hallazgos_sistema) else pd.DataFrame(
        columns=["solicitud", "tipo", "bs"])
    return {
        "Ordenes en la planilla": len(df),
        "Ordenes con riesgo ALTO (reglas sobre la planilla)": int((anomalias["nivel_riesgo"] == "ALTO").sum()),
        "Ordenes con riesgo MEDIO (reglas sobre la planilla)": int((anomalias["nivel_riesgo"] == "MEDIO").sum()),
        "PDFs del sistema leidos": info_pdf["ordenes_leidas"],
        "Ordenes de la planilla sin PDF (no comparadas)": info_pdf["ordenes_sin_pdf"],
        "PDFs escaneados o ilegibles": len(info_pdf["no_leidos"]),
        "PDFs cuya solicitud no esta en la planilla": len(info_pdf["fuera_de_planilla"]),
        "Ordenes con diferencias contra el sistema": int(hs["solicitud"].nunique()) if len(hs) else 0,
        "Diferencias contra el sistema (total)": len(hs),
        "  de ellas, redondeos (decimales)": int((hs["tipo"] == "REDONDEO").sum()) if len(hs) else 0,
        "Bs que dejarias de facturar": round(float(-hs.loc[hs["bs"] < 0, "bs"].sum()), 2) if len(hs) else 0.0,
        "Bs facturados de mas (riesgo de multa)": round(float(hs.loc[hs["bs"] > 0, "bs"].sum()), 2) if len(hs) else 0.0,
        "Avisos preventivos IPTV": len(alertas_iptv_df),
        "Ordenes con alerta bajada por posible FTTB": int((anomalias["nota_fttb"] != "").sum())
                                                       if "nota_fttb" in anomalias.columns else 0,
    }


def exportar_informe(df, anomalias, hallazgos_sistema, alertas_iptv_df, resumen_tec, info_pdf,
                     nombre_base="informe_conciliacion"):
    """Crea el Excel y el texto del informe. Devuelve (ruta_xlsx, ruta_txt, resumen, hallazgos)."""
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter

    resumen = resumen_general(df, anomalias, hallazgos_sistema, alertas_iptv_df, info_pdf)
    hallazgos = construir_hallazgos(anomalias, hallazgos_sistema, alertas_iptv_df)
    resumen_df = pd.DataFrame({"Indicador": list(resumen), "Valor": list(resumen.values())})
    con_nota = anomalias["nota_fttb"] != "" if "nota_fttb" in anomalias.columns else False
    con_alertas = anomalias[(anomalias["n_alertas"] > 0) | con_nota][
        ["solicitud", "tipo_trabajo", "tipo_servicio", "puntaje_riesgo", "nivel_riesgo", "detalle_alertas"]
        + (["nota_fttb"] if "nota_fttb" in anomalias.columns else [])]
    no_leidos = pd.DataFrame({
        "archivo_o_solicitud": list(info_pdf["no_leidos"]) + [str(s) for s in info_pdf["fuera_de_planilla"]],
        "motivo": ["PDF escaneado o sin texto"] * len(info_pdf["no_leidos"])
                  + ["La solicitud no esta en la planilla"] * len(info_pdf["fuera_de_planilla"])})
    hojas = {"Resumen": resumen_df, "Hallazgos": hallazgos,
             "Planilla vs sistema": hallazgos_sistema if len(hallazgos_sistema) else pd.DataFrame(
                 columns=["solicitud", "material", "planilla", "sistema", "tipo", "detalle", "bs", "efecto", "gravedad"]),
             "Ordenes con alertas": con_alertas, "Tecnicos": resumen_tec, "PDF no leidos": no_leidos}

    ruta_xlsx, ruta_txt = f"{nombre_base}.xlsx", f"{nombre_base}.txt"
    with pd.ExcelWriter(ruta_xlsx, engine="openpyxl") as w:
        for nombre, tabla in hojas.items():
            tabla.to_excel(w, sheet_name=nombre, index=False)
            ws = w.sheets[nombre]
            for c in ws[1]:
                c.font = Font(bold=True, color="FFFFFF")
                c.fill = PatternFill("solid", fgColor="1F4E78")
            ws.freeze_panes = "A2"
            if ws.max_row > 1:
                ws.auto_filter.ref = ws.dimensions
            for i, col in enumerate(tabla.columns, 1):
                largo = max([len(str(col))] + [len(str(v)) for v in tabla[col].head(200)])
                ws.column_dimensions[get_column_letter(i)].width = min(max(10, largo + 2), 70)
            for fila in ws.iter_rows(min_row=2):
                for c in fila:
                    c.alignment = Alignment(wrap_text=True, vertical="top")
        colores = {"ALTA": "F8CBAD", "MEDIA": "FFE699", "BAJA": "E2EFDA"}
        ws = w.sheets["Hallazgos"]
        for fila in ws.iter_rows(min_row=2):
            gr = fila[2].value
            if gr in colores:
                fila[2].fill = PatternFill("solid", fgColor=colores[gr])

    lineas = ["INFORME DE CONCILIACION - REVISION ANTES DE ENVIAR AL OPERADOR", "=" * 60, ""]
    lineas += [f"{k}: {v}" for k, v in resumen.items()]
    lineas += ["", "HALLAZGOS (ordenados por gravedad)", "-" * 60]
    for _, r in hallazgos.iterrows():
        mat = f" | {r['material']}: planilla={r['planilla']:g} sistema={r['sistema']:g}" if r["material"] else ""
        lineas.append(f"[{r['gravedad']}] {r['solicitud']} | {r['origen']}{mat}")
        lineas.append(f"      {r['descripcion']}")
        lineas.append(f"      -> {r['accion_sugerida']}")
    with open(ruta_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lineas) + "\n")
    return ruta_xlsx, ruta_txt, resumen, hallazgos


In [ ]:
# Si se omitio el paso de PDFs, el informe se arma solo con las reglas sobre la planilla
try:
    hallazgos_sistema, alertas_iptv_df, info_pdf
except NameError:
    hallazgos_sistema, alertas_iptv_df = pd.DataFrame(), pd.DataFrame()
    info_pdf = {"pdfs_en_carpeta": 0, "ordenes_leidas": 0, "no_leidos": [], "fuera_de_planilla": [],
                "ordenes_planilla": len(df), "ordenes_sin_pdf": len(df), "solicitudes_leidas": []}

ruta_xlsx, ruta_txt, resumen, hallazgos = exportar_informe(
    df, anomalias, hallazgos_sistema, alertas_iptv_df, resumen_por_tecnico(df), info_pdf
)
display(pd.DataFrame({"Indicador": list(resumen), "Valor": list(resumen.values())}))
print(f"Hallazgos en total: {len(hallazgos)}")
hallazgos.head(30)


In [ ]:
from google.colab import files

files.download(ruta_xlsx)
files.download(ruta_txt)


## Opcional - Machine Learning para la monografia
Estas dos celdas no hacen falta para el informe. Sirven para la fase de modelado y evaluacion.

- **Clasificador de riesgo (Random Forest):** aprende de que tecnico, zona y tipo de servicio se asocian a mas riesgo.
- **Evaluacion contra las observaciones reales de el operador:** mide cuantos errores observados por el operador atrapa el detector. Para que la medicion sea valida, sube la planilla en la version que se envio a el operador, no la corregida despues.

In [ ]:
def entrenar_clasificador_riesgo(anomalias: pd.DataFrame):
    """Entrena un Random Forest que predice el nivel de riesgo (BAJO/MEDIO/ALTO)
    a partir de caracteristicas del CONTEXTO de la orden (tipo de trabajo, tipo
    de servicio, zona, cantidad de IPTV, tecnico) -- es decir, variables que se
    conocen antes/independientemente de revisar material por material.

    Esto es complementario al motor de reglas: el motor de reglas te dice
    exactamente QUE esta mal en una orden ya cerrada; este modelo aprende que
    patrones (ej. cierto tecnico, cierta zona, cierto tipo de servicio) se
    asocian historicamente a mayor riesgo, util para priorizar revision incluso
    antes de que la orden este completamente cargada.
    """
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import classification_report

    datos = anomalias.copy()
    columnas_categoricas = ["tipo_trabajo", "tipo_servicio", "area_cobertura", "tecnico1"]

    X = pd.DataFrame(index=datos.index)
    codificadores = {}
    for col in columnas_categoricas:
        le = LabelEncoder()
        X[col] = le.fit_transform(datos[col].astype(str))
        codificadores[col] = le
    X["cantidad_iptv"] = datos["cantidad_iptv"]

    y = datos["nivel_riesgo"]

    modelo = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")

    # Dataset chico -> validacion cruzada (k=5) en vez de un solo train/test split
    n_clases_presentes = y.value_counts().min()
    k = min(5, n_clases_presentes) if n_clases_presentes >= 2 else 2
    cv = StratifiedKFold(n_splits=max(k, 2), shuffle=True, random_state=42)

    predicciones_cv = cross_val_predict(modelo, X, y, cv=cv)
    print("--- Evaluacion por validacion cruzada (k=%d) ---" % cv.get_n_splits())
    print(classification_report(y, predicciones_cv, zero_division=0))

    # Entrenamiento final con todos los datos, para usar en ordenes nuevas
    modelo.fit(X, y)

    importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("Importancia de variables:")
    print(importancias.to_string())

    return modelo, codificadores, importancias


modelo, codificadores, importancias = entrenar_clasificador_riesgo(anomalias)


In [ ]:
def preparar_observaciones(observaciones: list) -> pd.DataFrame:
    """Convierte la lista de observaciones del operador en tabla, con la diferencia
    (conciliacion - sistema) y su valor en Bs."""
    obs = pd.DataFrame(
        observaciones,
        columns=["solicitud", "material", "valor_conciliacion", "valor_sistema"],
    )
    obs["solicitud"] = obs["solicitud"].astype(int)
    obs["diferencia"] = obs["valor_conciliacion"] - obs["valor_sistema"]
    obs["diferencia_bs"] = obs.apply(
        lambda r: r["diferencia"] * PRECIOS_UNITARIOS.get(r["material"], 0.0), axis=1
    )
    return obs


def evaluar_detector(anomalias: pd.DataFrame, obs: pd.DataFrame):
    """Compara el nivel de riesgo calculado contra las ordenes que el operador observo.
    Devuelve (resumen, detalle). Ordenes no observadas se toman como sin error."""
    from sklearn.metrics import roc_auc_score, average_precision_score

    a = anomalias.copy()
    a["solicitud"] = a["solicitud"].astype(int)
    a["con_error"] = a["solicitud"].isin(set(obs["solicitud"])).astype(int)
    total = int(a["con_error"].sum())

    filas = []
    for nombre, cond in [("Cualquier alerta (MEDIO o ALTO)", a["n_alertas"] > 0),
                         ("Solo ALTO", a["nivel_riesgo"] == "ALTO")]:
        sel = a[cond]
        tp = int(sel["con_error"].sum())
        filas.append({
            "criterio": nombre,
            "ordenes_marcadas": len(sel),
            "errores_atrapados": f"{tp}/{total}",
            "recall": round(tp / total, 2) if total else 0,
            "precision": round(tp / len(sel), 3) if len(sel) else 0,
        })
    resumen = pd.DataFrame(filas)
    print(f"Tasa base de error: {total}/{len(a)} = {total / len(a):.1%}")
    print(f"AUC del puntaje de riesgo: {roc_auc_score(a['con_error'], a['puntaje_riesgo']):.2f} "
          f"(0.50 = azar) | precision promedio: "
          f"{average_precision_score(a['con_error'], a['puntaje_riesgo']):.3f}")

    detalle = []
    for sol, grupo in obs.groupby("solicitud"):
        fila = a[a["solicitud"] == sol]
        if fila.empty:
            detalle.append({"solicitud": sol, "nivel_riesgo": "NO ESTA EN LA PLANILLA"})
            continue
        fila = fila.iloc[0]
        mats = list(grupo["material"])
        detalle.append({
            "solicitud": sol,
            "materiales_con_diferencia": ", ".join(mats),
            "puntaje_riesgo": fila["puntaje_riesgo"],
            "nivel_riesgo": fila["nivel_riesgo"],
            "alerta_sobre_ese_material": any(m in fila["detalle_alertas"] for m in mats),
        })
    return resumen, pd.DataFrame(detalle)


def entrenar_clasificador_con_etiquetas_reales(df: pd.DataFrame, obs: pd.DataFrame, repeticiones: int = 10):
    """Random Forest entrenado con etiquetas REALES (orden observada por el operador = 1).
    Usa validacion cruzada estratificada repetida porque hay muy pocos positivos."""
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import RepeatedStratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.preprocessing import LabelEncoder

    d = df.copy()
    d["solicitud"] = d["solicitud"].astype(int)
    y = d["solicitud"].isin(set(obs["solicitud"])).astype(int).values
    positivos = int(y.sum())
    if positivos < 2:
        print("Muy pocos positivos para entrenar.")
        return None

    X = pd.DataFrame(index=d.index)
    for col in ["tipo_servicio", "area_cobertura", "tecnico1"]:
        X[col] = LabelEncoder().fit_transform(d[col].astype(str))
    for col in MATERIALES + ["postes", "cantidad_iptv"]:
        X[col] = d[col].astype(float)

    splits = min(5, positivos)
    cv = RepeatedStratifiedKFold(n_splits=splits, n_repeats=repeticiones, random_state=42)
    aucs, aps = [], []
    for tr, te in cv.split(X, y):
        rf = RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                    min_samples_leaf=3, random_state=0).fit(X.iloc[tr], y[tr])
        prob = rf.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y[te], prob))
        aps.append(average_precision_score(y[te], prob))

    print(f"Positivos: {positivos} de {len(y)} (tasa base {y.mean():.1%})")
    print(f"AUC: {np.mean(aucs):.2f} (+-{np.std(aucs):.2f}) | precision promedio: "
          f"{np.mean(aps):.3f} (tasa base {y.mean():.3f})")
    modelo = RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                    min_samples_leaf=3, random_state=0).fit(X, y)
    importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)
    return modelo, importancias


# Observaciones de el operador de septiembre 2026: (solicitud, material, valor en la planilla, valor en el sistema).
# Para otro mes, reemplaza esta lista. La orden 345060082 (falta conclusion en NGOSS) no es de materiales.
OBSERVACIONES_OPERADOR = [
    (344729951, "PATCH_CORD_SCSC", 1, 2),   (344768433, "CABLE_DROP", 240, 24),
    (344765011, "CABLE_UTP", 12, 13),       (345020287, "CABLE_DUCTO", 0, 8),
    (345022338, "TERMINAL_SC", 1, 2),       (345110157, "CABLE_DROP", 326, 226),
    (345135769, "CABLE_DUCTO", 0, 37),      (345135769, "CABLE_DROP", 37, 0),
    (345103049, "CABLE_UTP", 27, 8),        (345103049, "CONECTOR_RJ45", 4, 2),
    (345103049, "CABLE_HDMI", 2, 1),        (345103049, "ADAPTADOR_ACDC", 3, 2),
    (345156865, "CABLE_DUCTO", 12, 6),      (345157047, "CONECTOR_RJ45", 2, 4),
    (345161636, "TERMINAL_SC", 2, 1),       (345162842, "CABLE_UTP", 29, 39),
    (345162842, "CONECTOR_RJ45", 4, 6),     (345177128, "CABLE_DUCTO", 0, 6),
    (345182012, "CABLE_DROP", 258, 259),
]
obs = preparar_observaciones(OBSERVACIONES_OPERADOR)
resumen_eval, detalle_eval = evaluar_detector(anomalias, obs)
display(resumen_eval)
display(detalle_eval)
modelo_real, importancias_real = entrenar_clasificador_con_etiquetas_reales(df, obs)
importancias_real.head(10)


## Opcional - Comparacion de modelos (para la monografia)
Compara cuatro enfoques con la **misma** validacion cruzada estratificada repetida: puntaje por reglas (linea base), regresion logistica, Random Forest y XGBoost. Etiqueta: orden observada por el operador = 1. Se usan AUC y precision promedio porque solo ~3% de las ordenes son positivas. Ejecutala despues de la celda de evaluacion (usa `obs`).

In [ ]:
def comparar_modelos(df: pd.DataFrame, anomalias: pd.DataFrame, obs: pd.DataFrame, repeticiones: int = 10):
    """Compara 4 enfoques con la MISMA validacion cruzada estratificada repetida.
    Etiqueta: orden observada por el operador = 1. Metricas: AUC y precision promedio
    (no accuracy, porque solo ~3% de las ordenes son positivas)."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.model_selection import RepeatedStratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score
    from xgboost import XGBClassifier

    d = df.copy()
    d["solicitud"] = d["solicitud"].astype(int)
    y = d["solicitud"].isin(set(obs["solicitud"])).astype(int).values
    positivos = int(y.sum())
    if positivos < 2:
        print("Muy pocos positivos para comparar modelos.")
        return None

    X = pd.DataFrame(index=d.index)
    for col in ["tipo_servicio", "area_cobertura", "tecnico1"]:
        X[col] = LabelEncoder().fit_transform(d[col].astype(str))
    for col in MATERIALES + ["postes", "cantidad_iptv"]:
        X[col] = d[col].astype(float)

    puntaje = anomalias.assign(solicitud=anomalias["solicitud"].astype(int)) \
                       .set_index("solicitud")["puntaje_riesgo"].reindex(d["solicitud"]).fillna(0).values

    peso = (len(y) - positivos) / positivos   # desbalance para XGBoost

    def hacer(nombre):
        if nombre == "Regresion logistica":
            return make_pipeline(StandardScaler(),
                                 LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000))
        if nombre == "Random Forest":
            return RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                          min_samples_leaf=3, random_state=0)
        return XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, scale_pos_weight=peso, eval_metric="logloss",
                             random_state=0, n_jobs=1)

    nombres = ["Regresion logistica", "Random Forest", "XGBoost"]
    res = {n: ([], []) for n in nombres + ["Puntaje por reglas"]}
    cv = RepeatedStratifiedKFold(n_splits=min(5, positivos), n_repeats=repeticiones, random_state=42)
    for tr, te in cv.split(X, y):
        if y[te].sum() == 0:
            continue
        res["Puntaje por reglas"][0].append(roc_auc_score(y[te], puntaje[te]))
        res["Puntaje por reglas"][1].append(average_precision_score(y[te], puntaje[te]))
        for n in nombres:
            m = hacer(n).fit(X.iloc[tr], y[tr])
            p = m.predict_proba(X.iloc[te])[:, 1]
            res[n][0].append(roc_auc_score(y[te], p))
            res[n][1].append(average_precision_score(y[te], p))

    tabla = pd.DataFrame([{
        "modelo": n,
        "AUC": round(float(np.mean(a)), 2),
        "AUC_desv": round(float(np.std(a)), 2),
        "precision_promedio": round(float(np.mean(b)), 3),
    } for n, (a, b) in res.items()]).sort_values("AUC", ascending=False).reset_index(drop=True)
    print(f"Positivos: {positivos} de {len(y)} (tasa base {y.mean():.1%}); "
          f"AUC 0.50 = azar; precision promedio de azar = {y.mean():.3f}")
    return tabla


tabla_modelos = comparar_modelos(df, anomalias, obs)
display(tabla_modelos)


## Opcional - Exportar datos para Power BI y figuras del capitulo 4
Genera `Datos_PowerBI.xlsx` (tecnicos anonimizados) y las Figuras 4-1 a 4-5 en la carpeta `salida_anexos/`. Ejecutar despues de todas las celdas anteriores.

In [ ]:
# Genera (1) Datos_PowerBI.xlsx, con tecnicos anonimizados T01..Txx, y (2) las figuras 4-1 a 4-5.
# Se ejecuta al final, despues de haber corrido todo el notebook. Los resultados quedan en la carpeta salida_anexos/.
# La clave nombre real -> codigo NO se guarda en ningun archivo.
import os,glob,re,numpy as np,pandas as pd,openpyxl
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
df=df.copy(); an=anomalias.copy(); tabla=tabla_modelos
INF=ruta_xlsx
SALIDA='salida_anexos/'; os.makedirs(SALIDA+'figuras',exist_ok=True)
dif=pd.read_excel(INF,sheet_name='Planilla vs sistema'); hall=pd.read_excel(INF,sheet_name='Hallazgos')
df['solicitud']=df['solicitud'].astype(int); an['solicitud']=an['solicitud'].astype(int)
pdfs={int(os.path.basename(p)[:-4]) for p in glob.glob(os.path.join(CARPETA_PDFS,'**','*.pdf'),recursive=True)}
obs_set=set(obs['solicitud'].astype(int))
iptv={int(x) for x in hall.loc[hall['origen'].astype(str).str.contains('IPTV'),'solicitud']}
# ---- codigos de tecnico
res=resumen_por_tecnico(df)
cod={n:f'T{i+1:02d}' for i,n in enumerate(res['tecnico'])}
def anon(txt):
    txt=str(txt) if pd.notna(txt) else ''
    for n,c in sorted(cod.items(),key=lambda x:-len(x[0])): txt=txt.replace(n,c)
    return txt
MAT=MATERIALES
o=df[['solicitud','tipo_trabajo','tipo_servicio','area_cobertura','cantidad_iptv','postes','tecnico1','tecnico2']+MAT].copy()
o['tecnico1']=o['tecnico1'].map(lambda x:cod.get(str(x).strip(),'')); o['tecnico2']=o['tecnico2'].map(lambda x:cod.get(str(x).strip(),'') if x else '')
o=o.rename(columns={'area_cobertura':'zona','tecnico1':'tecnico1_cod','tecnico2':'tecnico2_cod'})
a=an[['solicitud','puntaje_riesgo','nivel_riesgo','n_alertas','detalle_alertas']].copy(); a['detalle_alertas']=a['detalle_alertas'].map(anon)
o=o.merge(a,on='solicitud',how='left')
o['observada_operador']=o['solicitud'].isin(obs_set).astype(int)
o['con_pdf']=o['solicitud'].isin(pdfs).astype(int)
o['con_diferencia_sistema']=o['solicitud'].isin(set(dif['solicitud'].astype(int))).astype(int)
o['aviso_iptv']=o['solicitud'].isin(iptv).astype(int)
dif2=dif.copy(); dif2['solicitud']=dif2['solicitud'].astype(int)
dif2['observada_operador']=dif2['solicitud'].isin(obs_set).astype(int)
tec=[]
for n,c in cod.items():
    m=(df['tecnico1'].astype(str).str.strip()==n)|(df['tecnico2'].astype(str).str.strip()==n)
    r=res[res['tecnico']==n].iloc[0]
    tec.append({'tecnico_cod':c,'instalaciones':int(r['instalaciones']),'adiciones':int(r['adiciones']),'total':int(r['total']),
                'ordenes_observadas':int(df.loc[m,'solicitud'].isin(obs_set).sum()),
                'ordenes_riesgo_alto':int(o.loc[m.values,'nivel_riesgo'].eq('ALTO').sum())})
tec=pd.DataFrame(tec)
P=PRECIOS_UNITARIOS
mat=pd.DataFrame({'material':MAT,'precio_bs':[P.get(m,0.0) for m in MAT]})
mat['tratamiento']=np.where(mat['precio_bs']>0,'con valor en la planilla del operador','sin valor (precio 0)')
mat['medido_en_metros']=mat['material'].isin(MATERIALES_METROS_ENTEROS).map({True:'si',False:'no'})
modelos=tabla.copy()
n=len(df); k=int(o['observada_operador'].sum())
kpi=pd.DataFrame([
 ['Periodo','2026-09'],['Ordenes en la planilla',n],['Ordenes observadas por el operador',k],['Tasa de observacion',round(k/n,4)],
 ['Ordenes riesgo ALTO',int((o.nivel_riesgo=='ALTO').sum())],['Ordenes riesgo MEDIO',int((o.nivel_riesgo=='MEDIO').sum())],
 ['Ordenes riesgo BAJO',int((o.nivel_riesgo=='BAJO').sum())],['Ordenes marcadas (MEDIO o ALTO)',int((o.nivel_riesgo!='BAJO').sum())],
 ['Carga de revision',round(float((o.nivel_riesgo!='BAJO').mean()),4)],
 ['Observadas con alerta de reglas',int(((o.nivel_riesgo!='BAJO')&(o.observada_operador==1)).sum())],
 ['PDF del sistema comparados',int(o.con_pdf.sum())],['Ordenes con diferencia vs sistema',int(o.con_diferencia_sistema.sum())],
 ['Bs que se dejarian de facturar',round(float(-dif2.loc[dif2.bs<0,'bs'].sum()),2)],['Bs facturados de mas',round(float(dif2.loc[dif2.bs>0,'bs'].sum()),2)],
 ['Avisos preventivos IPTV',len(iptv)]],columns=['indicador','valor'])
with pd.ExcelWriter(SALIDA+'Datos_PowerBI.xlsx') as w:
    o.to_excel(w, sheet_name='Ordenes',index=False); dif2.to_excel(w, sheet_name='Diferencias',index=False); tec.to_excel(w, sheet_name='Tecnicos',index=False)
    mat.to_excel(w, sheet_name='Materiales',index=False); modelos.to_excel(w, sheet_name='Modelos',index=False); kpi.to_excel(w, sheet_name='KPI',index=False)
print(kpi.to_string())
# ================= FIGURAS =================
INK='#0b0b0b'; INK2='#52514e'; MUTED='#898781'; GRID='#e1e0d9'; BASE='#c3c2b7'
B,OR,AQ,VI='#2a78d6','#eb6834','#1baf7a','#6250d6'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.edgecolor':BASE,'axes.labelcolor':INK2,'xtick.color':INK2,'ytick.color':INK2,
 'text.color':INK,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.color':GRID,'grid.linewidth':0.6,'axes.axisbelow':True})
OUT=SALIDA+'figuras/'
corto={'SOLO ONT':'Solo ONT','ONT + 1 STB':'ONT + 1 STB','ONT + 2 STB':'ONT + 2 STB','ONT + 3 STB':'ONT + 3 STB','ADICION DE SERVICIO':'Adición'}
orden=list(corto)
# 4-1 boxplots
mats=['CABLE_DROP','CABLE_DUCTO','CABLE_UTP','CONECTOR_RJ45','TERMINAL_SC','ANILLAS_GUIA']
tit={'CABLE_DROP':'Cable drop (m)','CABLE_DUCTO':'Cable ducto (m)','CABLE_UTP':'Cable UTP (m)','CONECTOR_RJ45':'Conector RJ45 (pza.)','TERMINAL_SC':'Terminal SC (pza.)','ANILLAS_GUIA':'Anillas guía (pza.)'}
fig,axs=plt.subplots(2,3,figsize=(6.6,4.6))
for ax,m in zip(axs.ravel(),mats):
    data=[df.loc[df.tipo_servicio==s,m].astype(float).values for s in orden]
    bp=ax.boxplot(data,patch_artist=True,widths=0.55,flierprops=dict(marker='o',ms=2.5,mfc=MUTED,mec='none',alpha=.8),
                  medianprops=dict(color=INK,lw=1.2),whiskerprops=dict(color=INK2,lw=.8),capprops=dict(color=INK2,lw=.8),boxprops=dict(lw=.8,ec=INK2))
    for p in bp['boxes']: p.set_facecolor('#9ec5f4')
    ax.set_title(tit[m],fontsize=8,color=INK,loc='left'); ax.set_xticks(range(1,6)); ax.set_xticklabels(['Solo\nONT','+1\nSTB','+2\nSTB','+3\nSTB','Adic.'],fontsize=6.5)
    ax.tick_params(axis='y',labelsize=7); ax.grid(axis='x',visible=False)
fig.tight_layout(); fig.savefig(OUT+'Figura_4-1_boxplots_materiales.png',dpi=300,facecolor='white'); plt.close(fig)
# 4-2 riesgo por servicio
tab=pd.crosstab(o['tipo_servicio'],o['nivel_riesgo']).reindex(orden)[['BAJO','MEDIO','ALTO']].fillna(0)
tab=tab.div(tab.sum(1),axis=0)*100; cnt=pd.crosstab(o['tipo_servicio'],o['nivel_riesgo']).reindex(orden)[['BAJO','MEDIO','ALTO']].fillna(0).astype(int)
cols={'BAJO':'#cde2fb','MEDIO':'#3987e5','ALTO':'#0d366b'}
fig,ax=plt.subplots(figsize=(6.4,2.9)); left=np.zeros(5)
for nv in ['BAJO','MEDIO','ALTO']:
    v=tab[nv].values; ax.barh(range(5),v,left=left,color=cols[nv],edgecolor='#fcfcfb',lw=1.5,height=.62,label=nv.capitalize())
    for i,(x,l,c) in enumerate(zip(v,left,cnt[nv].values)):
        if x>=6: ax.text(l+x/2,i,f'{c}',ha='center',va='center',fontsize=7,color='white' if nv!='BAJO' else INK)
    left+=v
ax.set_yticks(range(5)); ax.set_yticklabels([f"{corto[s]} (n={int(cnt.loc[s].sum())})" for s in orden]); ax.invert_yaxis(); ax.set_xlim(0,100)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{x:.0f} %')); ax.grid(axis='y',visible=False)
ax.legend(title='Nivel de riesgo',ncol=3,loc='upper center',bbox_to_anchor=(0.5,-0.14),frameon=False,fontsize=7,title_fontsize=7)
fig.tight_layout(); fig.savefig(OUT+'Figura_4-2_riesgo_por_servicio.png',dpi=300,facecolor='white'); plt.close(fig)
# 4-3 diferencias vs sistema
et={'REDONDEO':'Redondeo','DIFERENCIA DE 1':'Diferencia de 1','UN DIGITO DISTINTO':'Un dígito distinto','DIGITO DE MAS O DE MENOS':'Dígito de más o de menos','FALTA EN PLANILLA':'Falta en la planilla','SOBRA EN PLANILLA':'Sobra en la planilla','POSIBLE CAMBIO DE MATERIAL':'Posible cambio de material','CANTIDAD DISTINTA':'Cantidad distinta'}
g=dif2.assign(tipo=dif2['tipo'].map(et).fillna(dif2['tipo']))
c=g.groupby('tipo').size().sort_values(); 
bs=g.assign(dejas=np.where(g.bs<0,-g.bs,0),mas=np.where(g.bs>0,g.bs,0)).groupby('tipo')[['dejas','mas']].sum().reindex(c.index)
fig,axs=plt.subplots(1,2,figsize=(6.6,3.0),sharey=True,gridspec_kw={'width_ratios':[1,1.25]})
axs[0].barh(range(len(c)),c.values,color=B,height=.6); axs[0].set_yticks(range(len(c))); axs[0].set_yticklabels(c.index,fontsize=7)
for i,v in enumerate(c.values): axs[0].text(v+.15,i,str(v),va='center',fontsize=7,color=INK2)
axs[0].set_title('Diferencias (cantidad)',fontsize=8,loc='left'); axs[0].grid(axis='y',visible=False)
axs[1].barh(range(len(c)),bs['dejas'].values,color=OR,height=.6,label='No facturado'); axs[1].barh(range(len(c)),bs['mas'].values,left=bs['dejas'].values,color=VI,height=.6,edgecolor='#fcfcfb',lw=1.5,label='Facturado de más')
axs[1].set_title('Valor (Bs)',fontsize=8,loc='left'); axs[1].grid(axis='y',visible=False); axs[1].legend(frameon=False,fontsize=7,loc='lower right')
fig.tight_layout(); fig.savefig(OUT+'Figura_4-3_diferencias_planilla_vs_sistema.png',dpi=300,facecolor='white'); plt.close(fig)
# 4-4 y 4-5 modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score,average_precision_score,roc_curve,precision_recall_curve
from xgboost import XGBClassifier
d=df.copy(); y=d['solicitud'].isin(obs_set).astype(int).values; pos=int(y.sum())
X=pd.DataFrame(index=d.index)
for col in ['tipo_servicio','area_cobertura','tecnico1']: X[col]=LabelEncoder().fit_transform(d[col].astype(str))
for col in MAT+['postes','cantidad_iptv']: X[col]=d[col].astype(float)
punt=an.set_index('solicitud')['puntaje_riesgo'].reindex(d['solicitud']).fillna(0).values
peso=(len(y)-pos)/pos
def hacer(nm):
    if nm=='Regresión logística': return make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',C=0.1,max_iter=2000))
    if nm=='Random Forest': return RandomForestClassifier(n_estimators=300,class_weight='balanced_subsample',min_samples_leaf=3,random_state=0)
    return XGBClassifier(n_estimators=200,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,scale_pos_weight=peso,eval_metric='logloss',random_state=0,n_jobs=1)
nombres=['Regresión logística','Random Forest','XGBoost','Índice de riesgo (reglas)']
gx=np.linspace(0,1,101); R={n:{'tpr':[],'pr':[],'auc':[],'ap':[]} for n in nombres}
cv=RepeatedStratifiedKFold(n_splits=min(5,pos),n_repeats=10,random_state=42)
for tr,te in cv.split(X,y):
    if y[te].sum()==0: continue
    for nm in nombres:
        p=punt[te] if nm.startswith('Índice') else hacer(nm).fit(X.iloc[tr],y[tr]).predict_proba(X.iloc[te])[:,1]
        f,t,_=roc_curve(y[te],p); R[nm]['tpr'].append(np.interp(gx,f,t))
        pr,rc,_=precision_recall_curve(y[te],p); R[nm]['pr'].append(np.interp(gx,rc[::-1],pr[::-1]))
        R[nm]['auc'].append(roc_auc_score(y[te],p)); R[nm]['ap'].append(average_precision_score(y[te],p))
colm={'Regresión logística':B,'Random Forest':OR,'XGBoost':AQ,'Índice de riesgo (reglas)':VI}
fig,axs=plt.subplots(1,2,figsize=(6.6,3.2))
for nm in nombres:
    axs[0].plot(gx,np.mean(R[nm]['tpr'],0),color=colm[nm],lw=1.6,label=f"{nm} (AUC {np.mean(R[nm]['auc']):.2f})")
    axs[1].plot(gx,np.mean(R[nm]['pr'],0),color=colm[nm],lw=1.6,label=nm)
axs[0].plot([0,1],[0,1],ls='--',color=MUTED,lw=1); axs[0].set_xlabel('Tasa de falsos positivos'); axs[0].set_ylabel('Tasa de verdaderos positivos'); axs[0].set_title('Curva ROC',loc='left',fontsize=8)
axs[1].axhline(pos/len(y),ls='--',color=MUTED,lw=1); axs[1].text(0.98,pos/len(y)+.012,f'azar = {pos/len(y):.2f}',ha='right',fontsize=7,color=INK2)
axs[1].set_xlabel('Recall'); axs[1].set_ylabel('Precisión'); axs[1].set_title('Curva precisión-recall',loc='left',fontsize=8); axs[1].set_ylim(0,.5)
axs[0].legend(frameon=False,fontsize=6.3,loc='lower right')
fig.tight_layout(); fig.savefig(OUT+'Figura_4-5_curvas_ROC_PR.png',dpi=300,facecolor='white'); plt.close(fig)
fig,axs=plt.subplots(1,2,figsize=(6.6,2.7))
sd_auc=[np.std(R[n]['auc']) for n in nombres]; m_auc=[np.mean(R[n]['auc']) for n in nombres]; m_ap=[np.mean(R[n]['ap']) for n in nombres]
lab=['Regresión\nlogística','Random\nForest','XGBoost','Índice de\nriesgo']
axs[0].bar(range(4),m_auc,yerr=sd_auc,color=[colm[n] for n in nombres],width=.6,error_kw=dict(lw=.8,capsize=2,ecolor=INK2)); axs[0].axhline(.5,ls='--',color=MUTED,lw=1)
axs[0].text(3.45,.51,'azar',fontsize=7,color=INK2,ha='right'); axs[0].set_ylim(0,1); axs[0].set_title('AUC-ROC (media ± desv.)',loc='left',fontsize=8)
for i,v in enumerate(m_auc): axs[0].text(i,v+sd_auc[i]+.02,f'{v:.2f}',ha='center',fontsize=7)
axs[1].bar(range(4),m_ap,color=[colm[n] for n in nombres],width=.6); axs[1].axhline(pos/len(y),ls='--',color=MUTED,lw=1)
axs[1].text(3.45,pos/len(y)+.004,'azar',fontsize=7,color=INK2,ha='right'); axs[1].set_title('Precisión promedio',loc='left',fontsize=8)
for i,v in enumerate(m_ap): axs[1].text(i,v+.004,f'{v:.3f}',ha='center',fontsize=7)
for ax in axs: ax.set_xticks(range(4)); ax.set_xticklabels(lab,fontsize=6.8); ax.grid(axis='x',visible=False)
fig.tight_layout(); fig.savefig(OUT+'Figura_4-4_comparacion_modelos.png',dpi=300,facecolor='white'); plt.close(fig)
print({n:(round(np.mean(R[n]['auc']),3),round(np.std(R[n]['auc']),3),round(np.mean(R[n]['ap']),3)) for n in nombres})
print('OK post')
